In [1]:
import numpy as np
import pandas as pd
import pickle
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [2]:
df = pd.read_csv(r"C:\Users\ghema\Downloads\hindi_nlp_dataset_.csv")   # keep file in same folder

In [3]:
df.rename(columns={df.columns[0]: "text"}, inplace=True)

In [4]:
def clean_text(text):
    text = re.sub(r'[^\u0900-\u097F\s]', '', str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])

total_words = len(tokenizer.word_index) + 1
print("Vocab Size:", total_words)

Vocab Size: 71


In [6]:
input_sequences = []

for line in df['text']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [7]:
max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
input_sequences = np.array(input_sequences)

In [8]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

In [9]:
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(max_seq_len-1,)),
    Embedding(total_words, 100),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 7, 100)              │           7,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 150)                 │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 71)                  │          10,721 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 168,421 (657.89 KB)

 Trainable params: 168,421 (657.89 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(X, y, epochs=50, batch_size=128)

Epoch 1/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.8171 - loss: 0.8931
Epoch 2/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9868 - loss: 0.0312
Epoch 3/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9870 - loss: 0.0231
Epoch 4/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9877 - loss: 0.0209
Epoch 5/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9872 - loss: 0.0199
Epoch 6/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9869 - loss: 0.0196
Epoch 7/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9868 - loss: 0.0194
Epoch 8/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9864 - loss: 0.0192
Epoch 9/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9869 - loss: 0.0190
Epoch 10/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9872 - loss: 0.0192
Epoch 11/50
386/386 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.9866 - loss: 0.0189
Epoch 12/50
386/386 ━━━━━━━━━━━━━━━━━━━━

In [11]:
model.save("hindi_model.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("max_seq_len.pkl", "wb") as f:
    pickle.dump(max_seq_len, f)

In [12]:
model = load_model("hindi_model.keras")

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("max_seq_len.pkl", "rb") as f:
    max_seq_len = pickle.load(f)

In [13]:
def predict_top_words(text, top_n=3):
    text = clean_text(text)
    
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
    
    preds = model.predict(token_list)[0]
    top_indices = preds.argsort()[-top_n:][::-1]
    
    results = []
    for idx in top_indices:
        for word, index in tokenizer.word_index.items():
            if index == idx:
                results.append(word)
    return results

In [14]:
predict_top_words("भारत एक")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step


['सुंदर', 'देश', 'रोमांचक']